In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
# For each set of cell type exons, append exon # / transcript info.

In [3]:
intron_file_new = "data/intron_file_transcript.tab.gz"
intron_table_new = pd.read_csv(intron_file_new, sep='\t', index_col=0)
intron_table_new.columns = ["intron", "event", "gene", "transcript", "exon_number"]

In [4]:
intron_table_new.shape

(1905948, 5)

In [5]:
intron_table_new.head()

,intron,event,gene,transcript,exon_number
ENSG00000238009_other_1_I1,chr1:112805-120720:-,ENSG00000238009_other_1,ENSG00000238009,ENST00000477740,2.0
ENSG00000238009_other_1_I2,chr1:120933-129054:-,ENSG00000238009_other_1,ENSG00000238009,ENST00000477740,2.0
ENSG00000238009_other_1_SE,chr1:112805-129054:-,ENSG00000238009_other_1,ENSG00000238009,ENST00000477740,2.0
ENSG00000292994_other_1_I1,chr1:259026-261549:-,ENSG00000292994_other_1,ENSG00000292994,ENST00000448958,3.0
ENSG00000292994_other_1_I2,chr1:261635-267302:-,ENSG00000292994_other_1,ENSG00000292994,ENST00000448958,3.0


### First get coordinates for each exon (the same exon can get a different ID depending on its surrounding intron context)

In [ ]:
intron_file_old = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/psix_annotation/intron_file.tab.gz"
intron_table_old = pd.read_csv(intron_file_old, sep='\t', index_col=0)

In [7]:
intron_table_old[intron_table_old.index.str.contains("ENSG00000138326_ProteinCoding_1")]

,intron,event,gene
ENSG00000138326_ProteinCoding_1_I1,chr10:78037305-78037438:+,ENSG00000138326_ProteinCoding_1,ENSG00000138326
ENSG00000138326_ProteinCoding_1_I2,chr10:78037442-78037964:+,ENSG00000138326_ProteinCoding_1,ENSG00000138326
ENSG00000138326_ProteinCoding_1_SE,chr10:78037305-78037964:+,ENSG00000138326_ProteinCoding_1,ENSG00000138326


In [8]:
intron_table_old[intron_table_old.index.str.contains("ENSG00000138326_ProteinCoding_2")]

,intron,event,gene
ENSG00000138326_ProteinCoding_2_I1,chr10:78037305-78037438:+,ENSG00000138326_ProteinCoding_2,ENSG00000138326
ENSG00000138326_ProteinCoding_2_I2,chr10:78037442-78040203:+,ENSG00000138326_ProteinCoding_2,ENSG00000138326
ENSG00000138326_ProteinCoding_2_SE,chr10:78037305-78040203:+,ENSG00000138326_ProteinCoding_2,ENSG00000138326


In [10]:
"ENSG00000075711_NMD_3"

intron_table_old[intron_table_old.index.str.contains("ENSG00000075711_NMD_3")]

,intron,event,gene
ENSG00000075711_NMD_3_I1,chr3:197075871-197076585:-,ENSG00000075711_NMD_3,ENSG00000075711
ENSG00000075711_NMD_3_I2,chr3:197076686-197081050:-,ENSG00000075711_NMD_3,ENSG00000075711
ENSG00000075711_NMD_3_SE,chr3:197075871-197081050:-,ENSG00000075711_NMD_3,ENSG00000075711


In [7]:
intron_table_old.shape

(233295, 3)

In [8]:
intron_table_old.head()

,intron,event,gene
ENSG00000238009_other_1_I1,chr1:112805-120720:-,ENSG00000238009_other_1,ENSG00000238009
ENSG00000238009_other_1_I2,chr1:120933-129054:-,ENSG00000238009_other_1,ENSG00000238009
ENSG00000238009_other_1_SE,chr1:112805-129054:-,ENSG00000238009_other_1,ENSG00000238009
ENSG00000292994_other_1_I1,chr1:259026-261549:-,ENSG00000292994_other_1,ENSG00000292994
ENSG00000292994_other_1_I2,chr1:261635-267302:-,ENSG00000292994_other_1,ENSG00000292994


In [9]:
# Get coordinates for each exon (the same exon can get a different ID depending on its surrounding intron context)

intron_coords_df = intron_table_old['intron'].str.split(r"[:\-]", expand=True).iloc[:, :3]
intron_coords_df['intron'] = intron_table_old['intron']
intron_coords_df['gene'] = intron_table_old['gene']
intron_coords_df.columns = ["chr", "intron_first_base", "intron_last_base", "intron", "gene"]
intron_coords_df.index = intron_table_old.index
intron_coords_df['event'] = intron_coords_df.index.str.split("_").str[:3].str.join("_")
intron_coords_df['intron_first_base'] = intron_coords_df["intron_first_base"].astype(int)
intron_coords_df['intron_last_base'] = intron_coords_df["intron_last_base"].astype(int)

def safe_exon_coords(g):
    i1 = g.loc[g.index.str.contains("I1$"), "intron_last_base"].values
    i2 = g.loc[g.index.str.contains("I2$"), "intron_first_base"].values
    if len(i1) == 0 or len(i2) == 0:
        return pd.Series({"chr": None, "exon_start": None, "exon_end": None})
    return pd.Series({
        "chr": g["chr"].iloc[0],
        "exon_start": i1[0] + 1,
        "exon_end": i2[0] - 1,
    })

exon_coords_df = intron_coords_df.groupby("event").apply(safe_exon_coords)
exon_coords_df = exon_coords_df.dropna()  # drop exons with missing coords

In [10]:
intron_table_old.head()

,intron,event,gene
ENSG00000238009_other_1_I1,chr1:112805-120720:-,ENSG00000238009_other_1,ENSG00000238009
ENSG00000238009_other_1_I2,chr1:120933-129054:-,ENSG00000238009_other_1,ENSG00000238009
ENSG00000238009_other_1_SE,chr1:112805-129054:-,ENSG00000238009_other_1,ENSG00000238009
ENSG00000292994_other_1_I1,chr1:259026-261549:-,ENSG00000292994_other_1,ENSG00000292994
ENSG00000292994_other_1_I2,chr1:261635-267302:-,ENSG00000292994_other_1,ENSG00000292994


In [11]:
intron_coords_df.head()

,chr,intron_first_base,intron_last_base,intron,gene,event
ENSG00000238009_other_1_I1,chr1,112805,120720,chr1:112805-120720:-,ENSG00000238009,ENSG00000238009_other_1
ENSG00000238009_other_1_I2,chr1,120933,129054,chr1:120933-129054:-,ENSG00000238009,ENSG00000238009_other_1
ENSG00000238009_other_1_SE,chr1,112805,129054,chr1:112805-129054:-,ENSG00000238009,ENSG00000238009_other_1
ENSG00000292994_other_1_I1,chr1,259026,261549,chr1:259026-261549:-,ENSG00000292994,ENSG00000292994_other_1
ENSG00000292994_other_1_I2,chr1,261635,267302,chr1:261635-267302:-,ENSG00000292994,ENSG00000292994_other_1


In [12]:
# Add exon data to intron table
intron_exon_coords_df = intron_coords_df.merge(exon_coords_df[["exon_start", "exon_end"]], left_on="event", right_index=True, how="left")

In [13]:
intron_exon_coords_df.drop(columns=["intron_first_base", "intron_last_base"], inplace=True)

In [14]:
intron_exon_coords_df.head()

,chr,intron,gene,event,exon_start,exon_end
ENSG00000238009_other_1_I1,chr1,chr1:112805-120720:-,ENSG00000238009,ENSG00000238009_other_1,120721.0,120932.0
ENSG00000238009_other_1_I2,chr1,chr1:120933-129054:-,ENSG00000238009,ENSG00000238009_other_1,120721.0,120932.0
ENSG00000238009_other_1_SE,chr1,chr1:112805-129054:-,ENSG00000238009,ENSG00000238009_other_1,120721.0,120932.0
ENSG00000292994_other_1_I1,chr1,chr1:259026-261549:-,ENSG00000292994,ENSG00000292994_other_1,261550.0,261634.0
ENSG00000292994_other_1_I2,chr1,chr1:261635-267302:-,ENSG00000292994,ENSG00000292994_other_1,261550.0,261634.0


### Now merge with the new intron table to get transcript / exon number info.

In [15]:
# Now merge with the new intron table to get transcript / exon number info.
merged = intron_exon_coords_df.merge(
    intron_table_new[['gene', 'intron', 'transcript', 'exon_number']],
    on=['gene', 'intron'],
    how='left'
)
merged = merged.drop_duplicates()

In [16]:
merged.head()

,chr,intron,gene,event,exon_start,exon_end,transcript,exon_number
0,chr1,chr1:112805-120720:-,ENSG00000238009,ENSG00000238009_other_1,120721.0,120932.0,ENST00000477740,2.0
1,chr1,chr1:120933-129054:-,ENSG00000238009,ENSG00000238009_other_1,120721.0,120932.0,ENST00000477740,2.0
2,chr1,chr1:112805-129054:-,ENSG00000238009,ENSG00000238009_other_1,120721.0,120932.0,ENST00000477740,2.0
3,chr1,chr1:259026-261549:-,ENSG00000292994,ENSG00000292994_other_1,261550.0,261634.0,ENST00000448958,3.0
4,chr1,chr1:261635-267302:-,ENSG00000292994,ENSG00000292994_other_1,261550.0,261634.0,ENST00000448958,3.0


In [17]:
# Now collapse by event, joining transcript / exon number info with semicolons if there are multiple transcripts in which the event apperas

def join_pairs(group):
    group = group.drop_duplicates(subset=['transcript'])
    pairs = list(zip(group['transcript'].dropna(), group['exon_number'].dropna()))
    if not pairs:
        return pd.Series({'transcript': '', 'exon_number': ''})
    transcripts, exons = zip(*pairs)
    return pd.Series({
        'transcript': ';'.join(transcripts),
        'exon_number': ';'.join(str(int(e)) for e in exons)
    })

merged_agg = merged.groupby(['gene', 'chr', 'exon_start', 'exon_end', 'event']).apply(join_pairs).reset_index()

In [18]:
merged_agg.head()

,gene,chr,exon_start,exon_end,event,transcript,exon_number
0,ENSG00000000003,chrX,100632485.0,100632568.0,ENSG00000000003_ProteinCoding_1,ENST00000373020,6
1,ENSG00000000419,chr20,50940865.0,50940933.0,ENSG00000000419_ProteinCoding_1,ENST00000371588;ENST00000681979;ENST0000068236...,7;6;2;7;2;6;7;7
2,ENSG00000000419,chr20,50940865.0,50940955.0,ENSG00000000419_NMD_1,ENST00000371588;ENST00000681979;ENST0000068236...,7;6;2;7;2;6;4
3,ENSG00000000419,chr20,50941105.0,50941209.0,ENSG00000000419_ProteinCoding_2,ENST00000371584;ENST00000371582;ENST0000049475...,7;7;4;7;6;2
4,ENSG00000000419,chr20,50941105.0,50941209.0,ENSG00000000419_other_1,ENST00000494752;ENST00000371584;ENST0000037158...,4;7;7;7;2;6


### Now merge exon info with cell type exon results

In [50]:
column_order = ['gene', 'Gene', 'is_specific', 'specific_direction', 'exon_len', 
                'chr', 'exon_start', 'exon_end', 'transcript', 'exon_number', 
                'r', 'fdr', 
                'CGE Class', 'All GABAergic', 'All Neuronal',
                'Upper layer glutamatergic', 'Deep layer glutamatergic', 'Oligo', 'OPC',
                'Astro', 'Micro/PVM', 'VLMC', 'Endo', 'Peri'
                ]

In [57]:
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        print(file)
        signif_exons = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        signif_exons['event'] = signif_exons.index
        signif_exons_merged = merged_agg.merge(
            signif_exons, 
            on=['event', 'chr', 'exon_start', 'exon_end'],
            how='right'
        )
        signif_exons_merged.set_index('event', inplace=True, drop=True)
        rest_columns = signif_exons_merged.columns[signif_exons_merged.columns.str.contains("diff")].tolist() 
        new_file = file.replace('_exons.csv', '_exons_annotated.csv')
        signif_exons_merged[column_order + rest_columns].to_csv(f"data/ctype_exons/{new_file}")

Oligo_exons.csv
VLMC_exons.csv
Endo_exons.csv
Deep_layer_glutamatergic_exons.csv
Astro_exons.csv
OPC_exons.csv
Micro_PVM_exons.csv
All_Neuronal_exons.csv
All_GABAergic_exons.csv
Peri_exons.csv
CGE_Class_exons.csv
Upper_layer_glutamatergic_exons.csv
